In [ ]:
import anndata

example_sce = anndata.read_h5ad("data/million_cells.h5ad", backed=True, chunk_size=int(2e4))

We can set the flag below to True to run on the full, on-disk dataset. Otherwise, this notebook uses a small on-disk dataset, just so that it can run quickly.

In [ ]:
run_full = False

This dataset has 1.4 million cells. We'll read 20K cells into memory at a time.

In [ ]:
example_sce.obs

In [ ]:
from scdesigner.simulators import NegBinCopula

model = NegBinCopula(epochs=1, chunk_size=int(2e4), lr=0.01)

if run_full:
    model.fit(example_sce, "~ celltype")
else:
    example_sce = example_sce[:2000].to_memory()
    example_sce.X = example_sce.X.todense()
    model.fit(example_sce, "~ celltype")

In [ ]:
print(model.params["coef_dispersion"].values.max())
model.params["coef_dispersion"][model.params["coef_dispersion"]>80]=80

In [ ]:
from scdesigner.diagnose import compare_pca
import numpy as np

samples = model.sample(example_sce.obs.iloc[:2000, :])
compare_pca(example_sce[:2000].to_memory(), samples, np.log1p)